In [73]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

from scipy.sparse import vstack, load_npz
from sklearn.preprocessing import StandardScaler
import sklearn.metrics as skm
import sklearn.utils as sku

import torch as pt
import torch.nn as nn
import torch.optim as optim

from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader

import optuna as opt

import Basic_functions as bf

from torchao.sparsity.training import (
    SemiSparseLinear,
    SemiSparseActivationLinear,
    swap_linear_with_semi_sparse_linear,
    swap_semi_sparse_linear_with_linear,
)

# Optional imports
# from torch.nn import L1Loss
# from scipy.special import expit

In [ ]:
import numpy as np
from scipy import sparse
from pathlib import Path

import numpy as np
from scipy import sparse
from scipy.sparse import load_npz

# --------------------------------------------------------
# LOAD ARTICLE DATA + LABEL FILES
# --------------------------------------------------------

data_00 = load_npz("Data_files/data_articles_00.npz")
data_01 = load_npz("Data_files/data_articles_01.npz")
data_02 = load_npz("Data_files/data_articles_02.npz")
data_10 = load_npz("Data_files/data_articles_10.npz")
data_11 = load_npz("Data_files/data_articles_11.npz")
data_12 = load_npz("Data_files/data_articles_12.npz")
data_13 = load_npz("Data_files/data_articles_13.npz")
data_14 = load_npz("Data_files/data_articles_14.npz")
data_15 = load_npz("Data_files/data_articles_15.npz")
data_18 = load_npz("Data_files/data_articles_18.npz")

labels_article_00 = np.genfromtxt("Data_files/labels_articles_00.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_01 = np.genfromtxt("Data_files/labels_articles_01.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_02 = np.genfromtxt("Data_files/labels_articles_02.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_10 = np.genfromtxt("Data_files/labels_articles_10.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_11 = np.genfromtxt("Data_files/labels_articles_11.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_12 = np.genfromtxt("Data_files/labels_articles_12.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_13 = np.genfromtxt("Data_files/labels_articles_13.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_14 = np.genfromtxt("Data_files/labels_articles_14.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_15 = np.genfromtxt("Data_files/labels_articles_15.txt", delimiter="; ", dtype=str, encoding="latin1")
labels_article_18 = np.genfromtxt("Data_files/labels_articles_18.txt", delimiter="; ", dtype=str, encoding="latin1")

# --------------------------------------------------------
# CREATE COMBINED DATASET
# --------------------------------------------------------

data_articles_all = sparse.vstack(
    [data_00,
        data_01,
        data_02,
        data_10,
        data_11,
        data_12,
        data_13,
        data_14,
        data_15,
        data_18
    ],
    format="csr"
)

labels_articles_all = np.concatenate(
    [
        labels_article_00,
        labels_article_01,
        labels_article_02,
        labels_article_10,
        labels_article_11,
        labels_article_12,
        labels_article_13,
        labels_article_14,
        labels_article_15,
        labels_article_18
    ]
)

assert data_articles_all.shape[0] == len(labels_articles_all)

print("Final data shape:", data_articles_all.shape)
print("Number of labels:", len(labels_articles_all))

# --------------------------------------------------------
# index = 0, date 
# index = 1, whether its a sermon or not
# index = 2, article title
# index = 3, organization
# index = 4, article topic
# index = 5, lix score
# --------------------------------------------------------

def get_topic(line):
    parts = line.split(";")
    return parts[4].strip()

def get_lix_score(line):
    parts = line.split(";")
    return float(parts[5].strip())

# --------------------------------------------------------
# REMOVE TOPICS WITH FEWER THAN x OCCURRENCES
# --------------------------------------------------------

x = 100

topics = np.array([
    get_topic(line)
    for line in labels_articles_all
])

unique_topics, topic_counts = np.unique(topics, return_counts=True)
valid_topics = unique_topics[topic_counts >= x]

keep_mask = np.isin(topics, valid_topics)

print("Original number of articles:", data_articles_all.shape[0])
print("Filtered number of articles:", keep_mask.sum())
print("Removed articles:", data_articles_all.shape[0] - keep_mask.sum())
print("Topics kept:", len(valid_topics))

data_articles_all = data_articles_all[keep_mask]
labels_articles_all = labels_articles_all[keep_mask]

assert data_articles_all.shape[0] == len(labels_articles_all)

# --------------------------------------------------------
# ADD LIX SCORE AS LAST COLUMN IN DATA MATRIX
# --------------------------------------------------------

def get_lix_score(line):
    parts = line.split(";")

    if len(parts) >= 6:
        return float(parts[5].strip())

    return float(line.split()[-1].replace(";", ""))

lix_scores = np.array([
    get_lix_score(line)
    for line in labels_articles_all
], dtype=np.float32)

assert data_articles_all.shape[0] == len(lix_scores)

lix_column = sparse.csr_matrix(lix_scores.reshape(-1, 1))

data_articles_all = sparse.hstack(
    [lix_column,data_articles_all ],
    format="csr"
)

print("Final data shape:", data_articles_all.shape)
print(data_articles_all[0])
print(labels_articles_all[2])

ValueError: Some errors were detected !
    Line #18932 (got 3 columns instead of 5)
    Line #27425 (got 3 columns instead of 5)
    Line #42388 (got 3 columns instead of 5)

In [57]:
chars = "abcdefghijklmnopqrstuvwxyzæøå "
bigons = []
trigons = []
for i in range(len(chars)):
    for j in range(len(chars)):
        bigons.append(chars[i] + chars[j])
for i in range(len(chars)):
    for j in range(len(chars)):
        for k in range(len(chars)):
            trigons.append(chars[i] + chars[j] + chars[k])

# Making the training data

# Functions

In [58]:
class MyDataset(Dataset):    
    def __init__(self, X_data, y_data):
        self.input = X_data
        self.truth = y_data
        
    def __getitem__(self, index):
        return self.input[index], self.truth[index]
        
    def __len__ (self):
        return self.truth.shape[0]

#In pytorch, there is an additional step of turning your data into tensors
trained_data = MyDataset(data_train, data_train_labels)
val_data = MyDataset(data_val, data_val_labels)

# Define the model:
class TestModel(nn.Module):
    def __init__(self):
        super(TestModel, self).__init__()        # Here we define the layers.
        self.input_layer = nn.Linear(len(trigrams), 192)     #In pytorch, you define the input and output edges.
        self.hidden_layer1 = nn.Linear(192, 48)
        self.hidden_layer2 = nn.Linear(48, 12)
        self.output_layer = nn.Linear(12, 2)
        self.relu = nn.ReLU()
        
    def forward(self, inputs):                  # Here we define how data passes through the layers. 
        x = self.input_layer(inputs)            # Also here, pytorch is a bit more explicit in defining the layers and activation function separately
        x = self.relu(x)
        x = self.hidden_layer1(x)
        x = self.relu(x)
        x = self.hidden_layer2(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x
    
# Training loop:
def Train(model, optimizer, loss_function, train_loader, validation_loader, device, epochs):
    validation_loss = []
    training_loss   = []
    model.train()
    for e in range(0, epochs):
        epoch_loss = 0
        n_minibatches = 0
        for input_train_batch, truth_train_batch in train_loader:
            input_train_batch, truth_train_batch = input_train_batch.to(device), truth_train_batch.to(device)
            optimizer.zero_grad()
            prediction = model(input_train_batch)  # This asks our model to produce predictions on the training batch            
            loss = loss_function(prediction, truth_train_batch.long())  # This calculates the loss
            loss.backward()                                             # This initiates the backpropagation
            optimizer.step()
            epoch_loss += loss.item()
            n_minibatches += 1
        
        # Now that the model have trained 1 epoch, we evaluate the model on the validation set:
        valid_loss = Validate(model, validation_loader, device, loss_function)
        validation_loss.append(valid_loss)
        training_loss.append(epoch_loss/n_minibatches)
        print('EPOCH: %s | training loss: %s  | validation loss: %s'%(e+1,round(epoch_loss/n_minibatches,3), round(valid_loss, 3)))
    return training_loss, validation_loss


def Validate(model, validation_loader, device, loss_function):
    model.eval()
    n_batches  = 0
    validation_loss = 0
    with pt.no_grad():
        for input_valid_batch, truth_valid_batch in validation_loader:
            input_valid_batch, truth_valid_batch = input_valid_batch.to(device), truth_valid_batch.to(device)
            prediction = model(input_valid_batch)
            loss = loss_function(prediction, truth_valid_batch.long())
            validation_loss += loss.item()
            n_batches += 1
    validation_loss = validation_loss/n_batches
    return validation_loss


def Predict(model, prediction_loader, device):
    model.eval()
    predictions = []
    print('PREDICTING!')
    with pt.no_grad():
        for input_pred_batch, _ in prediction_loader:
            input_pred_batch = input_pred_batch.to(device)
            prediction = model(input_pred_batch)
            predictions.extend(prediction.numpy())
    print('Done Predicting!')
    return predictions
                
class MeanRelativeAbsoluteDeviationLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
                
    def forward(self, prediction, target):
        prediction = prediction.view_as(target)
        relative_absolute_error = pt.abs(prediction - target) / (pt.abs(target) + self.eps)
        return pt.mean(relative_absolute_error)
    
def sparse_collate(batch):
    xs, ys = zip(*batch)

    first = xs[0]
    n_features = first.shape[-1]

    batch_rows = []
    batch_cols = []
    batch_vals = []

    for i, x in enumerate(xs):
        x = x.tocoo()
        batch_rows.append(np.full_like(x.col, i, dtype=np.int64))
        batch_cols.append(x.col.astype(np.int64))
        batch_vals.append(x.data.astype(np.float32))

    indices = pt.tensor(
        np.vstack([np.concatenate(batch_rows), np.concatenate(batch_cols)]),
        dtype=pt.long
    )
    values = pt.tensor(np.concatenate(batch_vals), dtype=pt.float32)

    X = pt.sparse_coo_tensor(indices, values, size=(len(xs), n_features)).coalesce()
    y = pt.tensor(np.asarray(ys), dtype=pt.long)
    return X, y

In [59]:
data_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14859580 stored elements and shape (83693, 27001)>

# Training

In [66]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import sklearn.metrics as skm

from torch.utils.data import Dataset, DataLoader
import torch as pt
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

# ========================================================
# CHOOSE NUMBER OF TOPICS
# ========================================================

TOP_X = 20

# --------------------------------------------------------
# PREPARE LABELS
# --------------------------------------------------------

# Extract the topic/category from each label string
def get_topic(line):
    parts = line.split(";")
    return parts[3].strip()

# Create array of all article topics
topics_all = np.array([get_topic(line) for line in labels_articles_all])

# --------------------------------------------------------
# KEEP ONLY TOP X MOST FREQUENT TOPICS
# --------------------------------------------------------

# Count occurrences of each unique topic
unique_topics, topic_counts = np.unique(
    topics_all,
    return_counts=True
)

# Sort topics by frequency (largest first)
sort_idx = np.argsort(topic_counts)[::-1]

# Select the TOP_X most common topics
top_topics = unique_topics[sort_idx][:TOP_X]
top_counts = topic_counts[sort_idx][:TOP_X]

print(f"Top {TOP_X} topics:")
for topic, count in zip(top_topics, top_counts):
    print(f"{topic}: {count}")

# Keep only samples belonging to top topics
top_topic_mask = np.isin(topics_all, top_topics)

data_articles_top = data_articles_all[top_topic_mask]
topics_top = topics_all[top_topic_mask]

print("\nFiltered data shape:", data_articles_top.shape)
print("Filtered labels shape:", topics_top.shape)

# --------------------------------------------------------
# ENCODE LABELS
# --------------------------------------------------------

# Convert text labels into integer class IDs
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(topics_top)

print("\nClasses:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

# --------------------------------------------------------
# TRAIN / VALIDATION / TEST SPLIT
# --------------------------------------------------------

# First split:
# 75% train
# 25% temporary data (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    data_articles_top,
    y_all,
    test_size=0.25,
    random_state=42,
    stratify=y_all
)

# Second split:
# Split temporary data into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# --------------------------------------------------------
# DATASET
# --------------------------------------------------------

class SparseDataset(Dataset):

    # Store feature matrix and labels
    def __init__(self, X, y):
        self.X = X
        self.y = y

    # Return number of samples
    def __len__(self):
        return self.X.shape[0]

    # Return one sample
    def __getitem__(self, idx):

        # Convert sparse row to dense float32 vector
        x = self.X[idx].toarray().ravel().astype(np.float32)

        # Get label
        y = self.y[idx]

        # Convert to PyTorch tensors
        return pt.tensor(x), pt.tensor(y, dtype=pt.long)

# Training batches
train_loader = DataLoader(
    SparseDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

# Validation batches
val_loader = DataLoader(
    SparseDataset(X_val, y_val),
    batch_size=32,
    shuffle=False
)

# Test batches
test_loader = DataLoader(
    SparseDataset(X_test, y_test),
    batch_size=32,
    shuffle=False
)

# --------------------------------------------------------
# MODEL
# --------------------------------------------------------

class ArticleTopicClassifier(nn.Module):

    def __init__(self, input_dim, num_classes):
        super().__init__()

        # Simple feedforward neural network
        self.net = nn.Sequential(

            # First hidden layer
            nn.Linear(input_dim, 192),
            nn.ReLU(),

            # Second hidden layer
            nn.Linear(192, 48),
            nn.ReLU(),

            # Output layer
            nn.Linear(48, num_classes)
        )

    # Forward pass
    def forward(self, x):
        return self.net(x)

# Use GPU if available
device = "cuda" if pt.cuda.is_available() else "cpu"

# Number of input features
input_dim = data_articles_top.shape[1]

# Number of output classes
num_classes = len(label_encoder.classes_)

# Create model
model = ArticleTopicClassifier(input_dim, num_classes).to(device)

# Cross-entropy loss for classification
loss_fn = nn.CrossEntropyLoss()

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --------------------------------------------------------
# TRAINING
# --------------------------------------------------------

# Lists for storing losses over epochs
train_losses = []
val_losses = []

# Train for 10 epochs
for epoch in range(10):

    # Put model in training mode
    model.train()

    train_loss = 0

    # ---------------- TRAINING LOOP ----------------

    for X_batch, y_batch in train_loader:

        # Move data to GPU/CPU
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch)

        # Compute loss
        loss = loss_fn(logits, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate batch loss
        train_loss += loss.item()

    # Average training loss for this epoch
    avg_train_loss = train_loss / len(train_loader)

    # Store training loss
    train_losses.append(avg_train_loss)

    # ---------------- VALIDATION LOOP ----------------

    # Put model in evaluation mode
    model.eval()

    val_loss = 0

    # Disable gradient calculation
    with pt.no_grad():

        for X_batch, y_batch in val_loader:

            # Move data to device
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            logits = model(X_batch)

            # Compute validation loss
            loss = loss_fn(logits, y_batch)

            # Accumulate validation loss
            val_loss += loss.item()

    # Average validation loss
    avg_val_loss = val_loss / len(val_loader)

    # Store validation loss
    val_losses.append(avg_val_loss)

    # Print epoch results
    print(
        f"Epoch {epoch+1} | "
        f"Train loss: {avg_train_loss:.4f} | "
        f"Validation loss: {avg_val_loss:.4f}"
    )

# --------------------------------------------------------
# VALIDATION EVALUATION
# --------------------------------------------------------

# Set model to evaluation mode
model.eval()

all_preds = []
all_true = []

# Disable gradient calculations
with pt.no_grad():

    for X_batch, y_batch in val_loader:

        X_batch = X_batch.to(device)

        # Forward pass
        logits = model(X_batch)

        # Predicted class = highest score
        preds = pt.argmax(logits, dim=1).cpu().numpy()

        # Store predictions and true labels
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

# Print classification metrics
print(
    skm.classification_report(
        all_true,
        all_preds,
        target_names=label_encoder.classes_
    )
)

# --------------------------------------------------------
# PLOT TRAINING + VALIDATION LOSS
# --------------------------------------------------------

plt.figure(figsize=(8, 5))

# Plot training loss
plt.plot(
    train_losses,
    marker="o",
    label="Training loss"
)

# Plot validation loss
plt.plot(
    val_losses,
    marker="s",
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title(f"Training vs Validation loss — top {TOP_X} topics")

# Show legend
plt.legend()

# Show grid
plt.grid(True)

plt.show()

# --------------------------------------------------------
# CONFUSION MATRIX
# --------------------------------------------------------

# Compute confusion matrix
cm = skm.confusion_matrix(all_true, all_preds)

# Create display object
disp = skm.ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(12, 12))

disp.plot(
    ax=ax,
    xticks_rotation=90,
    values_format="d"
)

plt.title(f"Validation confusion matrix — top {TOP_X} topics")
plt.show()

Top 20 topics:
Internationalt: 21060
Danmark: 17384
Politik: 6052
Sport: 3867
Fodbold: 3535
Kultur: 3074
Virksomheder: 2280
Superligaen: 2212
Forbrug: 2154
Økonomi: 1855
Film og tv: 1818
Sundhed: 1625
Dagens tegning: 1577
Premier League: 1429
Medier: 1234
Ting jeg gjorde: 1181
Musik: 1147
Debat: 1039
Tennis: 1022
Bøger: 983

Filtered data shape: (76528, 27001)
Filtered labels shape: (76528,)

Classes: ['Bøger' 'Dagens tegning' 'Danmark' 'Debat' 'Film og tv' 'Fodbold'
 'Forbrug' 'Internationalt' 'Kultur' 'Medier' 'Musik' 'Politik'
 'Premier League' 'Sport' 'Sundhed' 'Superligaen' 'Tennis'
 'Ting jeg gjorde' 'Virksomheder' 'Økonomi']
Number of classes: 20
Epoch 1 | Train loss: 2.0544 | Validation loss: 1.6595
Epoch 2 | Train loss: 1.5215 | Validation loss: 1.4451
Epoch 3 | Train loss: 1.3898 | Validation loss: 1.4059
Epoch 4 | Train loss: 1.2896 | Validation loss: 1.3060
Epoch 5 | Train loss: 1.1999 | Validation loss: 1.2366
Epoch 6 | Train loss: 1.1207 | Validation loss: 1.1779


KeyboardInterrupt: 